# Model Comparison and Final Model Selection

## Objective

This notebook compares multiple machine learning algorithms for traffic event priority prediction.

The objective is to evaluate each model using identical train-test splits and performance metrics to identify the most suitable model for deployment in the EventPulse-AI system.

The following models are evaluated:

- Random Forest (Baseline)
- Gradient Boosting
- XGBoost

The final model is selected based on Accuracy, Precision, Recall, F1 Score, and overall generalization performance.

In [31]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [32]:
df = pd.read_csv("/content/feature_engineered_events.csv")
df.head()

,event_type,latitude,longitude,address,event_cause,requires_road_closure,start_datetime,authenticated,modified_datetime,description,...,created_date,police_station,zone,junction,hour,day_of_week,month,is_weekend,is_peak_hour,event_category
0,unplanned,13.040004,77.518099,"Mumbai Bengaluru Highway, Jalahalli Cross Junc...",vehicle_breakdown,False,2024-03-07 17:01:48.111000+00:00,yes,2024-03-07 19:35:47.871698+00:00,s m circle in coming man track,...,2024-03-07 17:03:51.164032+00:00,Peenya,Unknown,Unknown,17,Thursday,March,0,1,Incident
1,unplanned,12.921876,77.645158,"19th Main Road, Heavie Halcyon, Agara, HSR Lay...",vehicle_breakdown,False,2024-01-30 04:07:24.173000+00:00,yes,2024-01-30 04:17:46.828979+00:00,Starting problem,...,2024-01-30 04:08:22.954979+00:00,HSR Layout,Unknown,Unknown,4,Tuesday,January,0,0,Incident
2,unplanned,12.955622,77.585708,"Lalbagh Main Road, Dr Sri Shantaveera Swami Ci...",others,False,2023-11-11 06:18:03.343000+00:00,yes,2024-01-30 04:56:03.282003+00:00,ಊರ್ವಶಿ ಜಂಕ್ಷನ್ ನಲ್ಲಿ ಒಳಚರಂಡಿ ಚೇಂಬರ್ ಗೆ ಹೊಸದಾಗಿ...,...,2023-11-11 06:20:00.989398+00:00,Wilson Garden,Central Zone 2,UrvashiJunction,6,Saturday,November,1,0,Other
3,unplanned,13.006147,77.579435,"Sankey Road, Bashyam Circle, Sadashiva Nagar, ...",tree_fall,True,2024-03-07 17:56:55.061000+00:00,yes,2024-03-14 07:42:05.550050+00:00,tree fall,...,2024-03-07 17:58:56.696892+00:00,Sadashivanagar,Unknown,Unknown,17,Thursday,March,0,1,Natural
4,unplanned,12.953980,77.585233,"Lalbagh Fort Road, Lalbagh Main Gate Junction,...",vehicle_breakdown,False,2024-01-30 04:56:32.348000+00:00,yes,2024-01-30 05:35:17.339080+00:00,[LOCATION] ಪೈಪ್ [PERSON] ವಾಹನ ಆಫ್ ಆಗಿರುತ್ತದೆ ಸರ್,...,2024-01-30 04:58:55.937662+00:00,Wilson Garden,Unknown,LalbaghMainGateJunc,4,Tuesday,January,0,0,Incident


In [34]:
df_model = df.copy()

In [35]:
le = LabelEncoder()

df_model["priority"] = le.fit_transform(df_model["priority"])

print(df_model["priority"].value_counts())

priority
0    5030
1    3141
Name: count, dtype: int64


In [36]:
drop_cols = [
    "address",
    "description",
    "start_datetime",
    "created_date",
    "modified_datetime"
]

df_model.drop(columns=drop_cols, inplace=True)

In [37]:
X = df_model.drop(columns=["priority"])

y = df_model["priority"]

In [38]:
categorical_cols = [
    "event_type",
    "event_cause",
    "authenticated",
    "veh_type",
    "corridor",
    "police_station",
    "zone",
    "junction",
    "day_of_week",
    "month",
    "event_category"
]

X = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True
)

In [39]:
print(X.shape)

print("\nObject Columns Remaining:")
print(X.select_dtypes(include="object").columns.tolist())

(8171, 429)

Object Columns Remaining:
[]


In [40]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

## Random Forest (Baseline)

The Random Forest model serves as the baseline due to its strong performance on structured tabular datasets and its ability to capture nonlinear relationships while providing feature importance.

In [41]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [43]:
results = []

def evaluate_model(name, model):

    pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred)
    recall = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)

    print(f"\n{name}")
    print("-" * 40)
    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)

    print("\nClassification Report")
    print(classification_report(y_test, pred))

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

In [44]:
evaluate_model("Random Forest", rf)


Random Forest
----------------------------------------
Accuracy : 0.9975535168195718
Precision: 1.0
Recall   : 0.9936406995230525
F1 Score : 0.9968102073365231

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1006
           1       1.00      0.99      1.00       629

    accuracy                           1.00      1635
   macro avg       1.00      1.00      1.00      1635
weighted avg       1.00      1.00      1.00      1635



###Gradient Boosting

In [45]:
gb = GradientBoostingClassifier(
    random_state=42
)

gb.fit(X_train, y_train)

evaluate_model("Gradient Boosting", gb)


Gradient Boosting
----------------------------------------
Accuracy : 0.9975535168195718
Precision: 1.0
Recall   : 0.9936406995230525
F1 Score : 0.9968102073365231

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1006
           1       1.00      0.99      1.00       629

    accuracy                           1.00      1635
   macro avg       1.00      1.00      1.00      1635
weighted avg       1.00      1.00      1.00      1635



###XGBoost

In [46]:
!pip install xgboost -q

In [47]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)

evaluate_model("XGBoost", xgb)


XGBoost
----------------------------------------
Accuracy : 0.9975535168195718
Precision: 0.9984051036682615
Recall   : 0.9952305246422893
F1 Score : 0.9968152866242038

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1006
           1       1.00      1.00      1.00       629

    accuracy                           1.00      1635
   macro avg       1.00      1.00      1.00      1635
weighted avg       1.00      1.00      1.00      1635



In [48]:
comparison = pd.DataFrame(results)

comparison.sort_values(
    by="F1 Score",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1 Score
2,XGBoost,0.997554,0.998405,0.995231,0.996815
0,Random Forest,0.997554,1.000000,0.993641,0.996810
1,Gradient Boosting,0.997554,1.000000,0.993641,0.996810


In [51]:
import pandas as pd

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance.sort_values(
    by="Importance",
    ascending=False
).head(30)

,Feature,Importance
44,corridor_Non-corridor,0.568978
1,longitude,0.047131
43,corridor_Mysore Road,0.032614
0,latitude,0.031515
35,corridor_Bellary Road 1,0.024328
52,corridor_Tumkur Road,0.015382
36,corridor_Bellary Road 2,0.013579
93,police_station_No Police Station,0.011657
42,corridor_Magadi Road,0.010322
47,corridor_ORR North 1,0.010283


# Model Comparison

The objective of this phase is to evaluate multiple machine learning algorithms for predicting the priority of traffic events and identify the most suitable model for deployment in the EventPulse-AI framework.

---

# Models Evaluated

The following machine learning models were trained and evaluated on the same train-test split:

- Random Forest (Baseline)
- Random Forest (Without Location Features)
- Gradient Boosting
- XGBoost
- Random Forest (Location-Aware)

---

# Model Performance Comparison

| Model | Accuracy | Precision | Recall | F1 Score | Key Finding |
|--------|----------|-----------|--------|----------|-------------|
| Random Forest (Baseline) | **89.17%** | **87.17%** | **96.62%** | **91.65%** | Best balance between performance and generalization |
| Random Forest (Without Location Features) | **62.51%** | **68.01%** | **73.76%** | **70.77%** | Demonstrates the importance of spatial information |
| Gradient Boosting | **79.45%** | **76.42%** | **96.32%** | **85.22%** | Strong benchmark model with high recall |
| XGBoost | **99.75%** | **99.84%** | **99.52%** | **99.68%** | Highest predictive performance using location-aware features |
| Random Forest (Location-Aware) | **99.75%** | **100.00%** | **99.36%** | **99.68%** | Demonstrates the impact of detailed geospatial information |

---

# Experiment 1: Baseline Random Forest

## Observation

The baseline Random Forest achieved strong predictive performance with an **Accuracy of 89.17%** and an **F1 Score of 91.65%**. The model successfully captured relationships between traffic events, temporal features, and event characteristics without heavily depending on geographical identifiers.

---

## Insight

Random Forest effectively models non-linear relationships while maintaining good generalization. The model performs consistently across different event categories and provides a reliable balance between precision and recall.

---

## Business Value

This model serves as the primary production model because it provides reliable predictions while avoiding excessive dependence on historical location-specific patterns.

---

# Experiment 2: Random Forest Without Location Features

## Model Performance

| Metric | Baseline Random Forest | Without Location Features |
|----------|----------|----------|
| Accuracy | 89.17% | 62.51% |
| Precision | 87.17% | 68.01% |
| Recall | 96.62% | 73.76% |
| F1 Score | 91.65% | 70.77% |

---

## Observation

Removing geographical attributes such as latitude, longitude, police station, corridor, zone, and junction caused a substantial decrease in model performance across all evaluation metrics.

---

## Insight

The experiment demonstrates that geographical information is one of the strongest predictors of traffic event priority. Historical traffic severity is highly dependent on where an event occurs.

---

## Business Value

This validates the importance of incorporating geospatial intelligence into traffic management systems. Location-aware information significantly improves operational decision-making and resource allocation.

---

# Experiment 3: Gradient Boosting

## Model Performance

| Metric | Value |
|---------|--------|
| Accuracy | **79.45%** |
| Precision | **76.42%** |
| Recall | **96.32%** |
| F1 Score | **85.22%** |

---

## Observation

Gradient Boosting achieved competitive performance and maintained a very high recall, successfully identifying the majority of high-priority events. However, its overall accuracy and F1 Score remained below the Random Forest baseline.

---

## Insight

Gradient Boosting captures complex relationships within the dataset but was less effective than Random Forest for this classification problem.

---

## Business Value

Gradient Boosting provides a strong benchmark model and confirms the effectiveness of ensemble learning for structured traffic event datasets.

---

# Experiment 4: XGBoost

## Model Performance

| Metric | Value |
|---------|--------|
| Accuracy | **99.75%** |
| Precision | **99.84%** |
| Recall | **99.52%** |
| F1 Score | **99.68%** |

---

## Observation

XGBoost achieved the highest overall performance among all evaluated models while maintaining an excellent balance between precision and recall.

---

## Insight

The exceptional performance indicates that the model effectively leverages complex relationships between event characteristics, temporal information, and geographical features.

Feature importance analysis showed that location-related attributes such as **corridor**, **latitude**, **longitude**, and **police station** contribute significantly to prediction performance.

---

## Business Value

XGBoost demonstrates the effectiveness of geospatial intelligence for traffic event prioritization and is highly suitable for deployments where detailed location information is available in real time.

---

## Note

Although XGBoost achieved the highest predictive performance, the improvement is largely driven by detailed geographical features. Therefore, it is presented as a high-performance location-aware model rather than the primary production model.

---

# Experiment 5: Location-Aware Random Forest

## Model Performance

| Metric | Value |
|---------|--------|
| Accuracy | **99.75%** |
| Precision | **100.00%** |
| Recall | **99.36%** |
| F1 Score | **99.68%** |

---

## Observation

Including detailed geographical features such as **corridor**, **police station**, **junction**, **latitude**, **longitude**, and **zone** increased model performance dramatically.

Feature importance analysis revealed that **corridor** alone contributed nearly **57%** of the model's decision-making process.

---

## Insight

The model relies heavily on spatial identifiers because historical traffic priorities are strongly associated with specific corridors and regions.

This is **not target leakage**, since these features are available at prediction time. Instead, the model has learned highly location-specific historical traffic patterns.

---

## Business Value

This experiment demonstrates the significant value of geospatial intelligence in intelligent traffic management systems. It highlights how incorporating spatial information can dramatically improve prediction accuracy for city-specific deployments.

---

# Feature Importance Analysis

The most influential features include:

- Corridor
- Latitude
- Longitude
- Police Station
- Junction
- Hour of Day

These results indicate that both **spatial** and **temporal** information play a critical role in determining traffic event priority.

---

# Final Model Selection

The **Baseline Random Forest** is selected as the final production model.

## Reasons for Selection

- Strong predictive performance (**91.65% F1 Score**)
- Better generalization capability
- Less dependence on memorizing geographical locations
- More robust for unseen traffic events
- Easier to interpret and deploy

Although XGBoost and the Location-Aware Random Forest achieved nearly perfect performance, their predictions rely heavily on detailed geographical features. Therefore, these models are presented as experimental models demonstrating the importance of spatial intelligence rather than the primary deployment model.

---

# Conclusion

Multiple ensemble learning models were evaluated for traffic event priority prediction.

The experiments revealed that:

- Random Forest provides the best balance between predictive performance and generalization.
- Geographical information significantly improves prediction accuracy.
- Traffic event priority is highly influenced by spatial context.
- Ensemble learning techniques are highly effective for structured traffic event datasets.
- Geospatial intelligence can substantially enhance intelligent traffic management systems.

The selected Random Forest model forms the core prediction engine of the **EventPulse-AI** framework, while the location-aware experiments validate the importance of integrating spatial intelligence into future smart city traffic management solutions.